In [19]:
import healpy as hp
from scripts.utils import plot_cl_alm, load_data
from scripts.config import SimConfig

import matplotlib.pyplot as plt
from pixell import enmap, lensing, curvedsky, reproject
import numpy as np

In [20]:
# from scripts.utils import get_radii
from ksw import Cosmology, Data
import camb
from ksw.radial_functional import radial_func
from tqdm.auto import tqdm

from scipy.interpolate import CubicSpline
from joblib import Parallel, delayed

In [22]:
t_scale = (s.cosmo_params['TCMB'] * 10**(6)) 
lhdus = (1, 2, 3) if s.npol > 1 else 1

fnls = []
alms = []
for i in np.arange(1, 1000):
    idx = str(i).zfill(4)
    base1 = f"scraped/alm_l_{idx}_v3.fits"
    base2 = f"scraped/alm_nl_{idx}_v3.fits"

    alm_heidelberg_l = hp.read_alm(base1, hdu=lhdus) * t_scale
    alm_heidelberg_nl = hp.read_alm(base2, hdu=lhdus) * t_scale

    fnl = np.random.normal(-1000, 1000)
    fnls.append(fnl)
    alms.append(alm_heidelberg_l + fnl * alm_heidelberg_nl)

In [23]:
lmax = 1024
ells = np.arange(2, lmax)
scale = ells * (ells + 1) / 2 / np.pi

In [26]:
import os
import sys
import numpy as np

import healpy as hp
import camb
import datetime

from ksw import Shape, KSW, Cosmology, Data
from astropy import units as u

from scripts.utils import load_data, save_data, get_radii, load_single_data
from scripts.config import SimConfig

# This will still crash on mainframe if MPI fails to start correctly....
# but try except doesn't work for some reason, and makes completion error
# from mpi4py import MPI

comm = None #MPI.COMM_WORLD
rank = 0 #comm.Get_rank()

In [29]:
config_file = "settings/scraped.json"
s = SimConfig(config_file, print_settings=(rank == 0))

camb_params_obj = camb.set_params(**s.cosmo_params)
cosmo = Cosmology(camb_params_obj)
cosmo.compute_transfer(s.cosmo_params["max_l"], verbose=s.verbose)
cosmo.compute_c_ell()

radii, drs = get_radii(s.settings["r_min"], s.settings["r_max"])
loc_shape = Shape.prim_local(
    ns=s.cosmo_params["ns"], pivot=s.cosmo_params["pivot_scalar"]
)
cosmo.add_prim_reduced_bispectrum(loc_shape, radii)

noise_ell, beam_ell = s.get_noise_beam()
data = Data(s.lmax, noise_ell, beam_ell, s.polarizations, cosmo)
icov = data.icov_diag_lensed if s.lensing else data.icov_diag_nonlensed

if s.disable_noise:
    def beam(alm):
        return alm  # hp.sphtfunc.smoothalm(alm, fwhm=0, pol=False)
else:
    beam_width = s.settings["beam_width"] * u.arcmin
    beam_width_rad = beam_width.to_value(u.radian)

    def beam(alm):
        return hp.sphtfunc.smoothalm(alm, fwhm=beam_width_rad)

ksw = KSW(
    cosmo.red_bispectra,
    icov,
    beam,
    s.lmax,
    s.polarizations,
    precision="double" if s.double_precision else "single",
)

theta_batch = 25  # nelem // 10000
alm_strs = np.arange(1, 1000).astype(str)
alm_strs = np.random.choice(alm_strs, size=100, replace=False)

def alm_loader(idx):
    return alms[int(idx) - 1]

ksw.step_batch(alm_loader, alm_strs, comm, verbose=False, theta_batch=theta_batch)

fisher = ksw.compute_fisher()
alm_strs = np.arange(1, 1000).astype(str)
estimates = ksw.compute_estimate_batch(
    alm_loader,
    alm_strs,
    comm,
    verbose=s.verbose,
    fisher=fisher,
    theta_batch=theta_batch,
)



Loaded settings from file: settings/scraped.json
{ 'alm_cache_dir': 'data/alm_cache',
  'base_dir': 'data',
  'beam_width': 7.1,
  'cosmo_params': { 'AccuracyBoost': 2.0,
                    'As': 2.457e-09,
                    'DoLateRadTruncation': False,
                    'H0': 70.1,
                    'TCMB': 2.7255,
                    'lAccuracyBoost': 2.0,
                    'lSampleBoost': 2.0,
                    'lens_potential_accuracy': 2,
                    'lmax': 1024,
                    'max_l': 1500,
                    'mnu': 0.06,
                    'ns': 0.96,
                    'ombh2': 0.02256,
                    'omch2': 0.1143,
                    'pivot_scalar': 0.05,
                    'r': 0,
                    'tau': 0.084},
  'debug': True,
  'disable_noise': True,
  'double_precision': False,
  'fnl_range': [0, 0],
  'force_alm_gen': True,
  'lensing': False,
  'name': 'scraped',
  'narray': 1,
  'noise_scale_ee': 4.3,
  'noise_scale_te': 0.43,


KeyboardInterrupt: 